# XGBoost Model — Extracted from `final_all_6_models.ipynb`

This notebook contains **only** the XGBoost model, copied exactly (unmodified) from the original six-model comparison notebook: the data load, the stratified train/test split, and the XGBoost pipeline + GridSearchCV cells. Supporting imports that were originally defined earlier in the source notebook (`SMOTE`, `Pipeline`, `GridSearchCV`) are added here so this notebook can run standalone.


In [2]:
import pandas as pd

# Load the FINAL combined dataset
df = pd.read_csv("maternal_modeling_common_features.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nClass distribution:")
print(df["RiskLevel"].value_counts())

print("\nClass distribution (%):")
print((df["RiskLevel"].value_counts(normalize=True) * 100).round(1))

df.head()


Dataset shape: (1880, 10)

Columns:
['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate', 'PulsePressure', 'MeanArterialPressure', 'RiskLevel', 'data_source']

Missing values: 0
Duplicate rows: 0

Class distribution:
RiskLevel
low risk     780
mid risk     623
high risk    477
Name: count, dtype: int64

Class distribution (%):
RiskLevel
low risk     41.5
mid risk     33.1
high risk    25.4
Name: proportion, dtype: float64


,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,PulsePressure,MeanArterialPressure,RiskLevel,data_source
0,25,130,80,15.00,98.0,86,50,96.666667,high risk,original
1,35,140,90,13.00,98.0,70,50,106.666667,high risk,original
2,29,90,70,8.00,100.0,80,20,76.666667,high risk,original
3,30,140,85,7.00,98.0,70,55,103.333333,high risk,original
4,23,140,80,7.01,98.0,70,60,100.000000,high risk,original


### 1.4 stratified train/test split


In [ ]:
from sklearn.model_selection import train_test_split

# Separate predictors and target.
# IMPORTANT: data_source is metadata only and must NOT be used as a model feature.
features = [
    "Age",
    "SystolicBP",
    "DiastolicBP",
    "BS",
    "BodyTemp",
    "HeartRate",
    "PulsePressure",
    "MAP"
]

X = df[features]
y = df["RiskLevel"]

# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Features used for modeling:")
print(X.columns.tolist())

print("\nTraining shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nOverall class distribution:")
print(y.value_counts(normalize=True))

print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))

assert "data_source" not in X.columns
assert "RiskLevel" not in X.columns


Features used for modeling:
['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate', 'PulsePressure', 'MeanArterialPressure']

Training shape: (1504, 8)
Test shape: (376, 8)

Overall class distribution:
RiskLevel
low risk     0.414894
mid risk     0.331383
high risk    0.253723
Name: proportion, dtype: float64

Training class distribution:
RiskLevel
low risk     0.414894
mid risk     0.331117
high risk    0.253989
Name: proportion, dtype: float64

Test class distribution:
RiskLevel
low risk     0.414894
mid risk     0.332447
high risk    0.252660
Name: proportion, dtype: float64


stratify=y tells Python when you split the dataset, try to keep the same proportion of low-, mid-, and high-risk cases in the training and test sets.

### Supporting imports

`SMOTE`, `Pipeline`, and `GridSearchCV` were imported earlier in the original notebook (in the Logistic Regression / Random Forest sections). Added here, unchanged, so this section is runnable on its own.


In [7]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV


### 2.6 XGBoost

XGBoost is trained under the same split and evaluation protocol. Labels are encoded only for XGBoost training and converted back to the original risk labels for evaluation.


In [9]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: xgboost in d:\anaconda\lib\site-packages (3.4.1)



In [10]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# XGBoost requires encoded class labels.
xgb_label_encoder = LabelEncoder()
xgb_label_encoder.fit(y_train)

y_train_xgb = xgb_label_encoder.transform(y_train)

xgb_pipeline = Pipeline([
    ("smote", SMOTE(random_state=42)),
    ("xgb", XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=1
    ))
])

param_grid_xgb = {
    "xgb__n_estimators": [100, 200],
    "xgb__max_depth": [3, 5, 7],
    "xgb__learning_rate": [0.03, 0.1],
    "xgb__subsample": [0.8, 1.0]
}

grid_search_xgb = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid_xgb,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid_search_xgb.fit(X_train, y_train_xgb)
best_xgb_model = grid_search_xgb.best_estimator_

y_pred_xgb_encoded = best_xgb_model.predict(X_test).astype(int)
y_pred_xgb = xgb_label_encoder.inverse_transform(y_pred_xgb_encoded)

print("Best XGBoost parameters:", grid_search_xgb.best_params_)
print("Best CV Macro-F1:", grid_search_xgb.best_score_)


Best XGBoost parameters: {'xgb__learning_rate': 0.03, 'xgb__max_depth': 5, 'xgb__n_estimators': 200, 'xgb__subsample': 0.8}
Best CV Macro-F1: 0.6127499642172162


## Model Evaluation — XGBoost

Same evaluation logic as the original notebook's Model Evaluation section, scoped to the XGBoost predictions only.


In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import pandas as pd


In [13]:
predictions = {
    "XGBoost": y_pred_xgb,
}

results = []

for model_name, y_pred in predictions.items():
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Macro Precision": precision,
        "Macro Recall": recall,
        "Macro F1": macro_f1
    })

results_df = pd.DataFrame(results)
results_df


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,XGBoost,0.648936,0.636628,0.644652,0.637035


In [14]:
print("Confusion Matrix (XGBoost):")
cm = confusion_matrix(
    y_test,
    y_pred_xgb,
    labels=["low risk", "mid risk", "high risk"]
)
print(cm)

print("\nClassification Report (XGBoost):")
print(classification_report(y_test, y_pred_xgb, zero_division=0))


Confusion Matrix (XGBoost):
[[121  25  10]
 [ 46  54  25]
 [  7  19  69]]

Classification Report (XGBoost):
              precision    recall  f1-score   support

   high risk       0.66      0.73      0.69        95
    low risk       0.70      0.78      0.73       156
    mid risk       0.55      0.43      0.48       125

    accuracy                           0.65       376
   macro avg       0.64      0.64      0.64       376
weighted avg       0.64      0.65      0.64       376



In [15]:
high_risk_results = []

for model_name, y_pred in predictions.items():

    high_risk_recall = recall_score(
        y_test,
        y_pred,
        labels=["high risk"],
        average=None,
        zero_division=0
    )[0]

    high_risk_results.append({
        "Model": model_name,
        "High-Risk Recall": high_risk_recall
    })


high_risk_df = pd.DataFrame(high_risk_results)

high_risk_df


,Model,High-Risk Recall
0,XGBoost,0.726316


In [16]:
high_risk_fnr_results = []

for model_name, y_pred in predictions.items():

    actual_high = (y_test == "high risk")
    predicted_high = (y_pred == "high risk")

    # High-risk true positives
    TP = ((actual_high) & (predicted_high)).sum()

    # High-risk false negatives:
    # actually high risk but predicted as another class
    FN = ((actual_high) & (~predicted_high)).sum()

    false_negative_rate = FN / (TP + FN)

    high_risk_fnr_results.append({
        "Model": model_name,
        "High-Risk True Positives": TP,
        "High-Risk False Negatives": FN,
        "High-Risk False Negative Rate": false_negative_rate
    })


high_risk_fnr_df = pd.DataFrame(high_risk_fnr_results)

high_risk_fnr_df


,Model,High-Risk True Positives,High-Risk False Negatives,High-Risk False Negative Rate
0,XGBoost,69,26,0.273684


In [17]:
final_results_df = results_df.merge(
    high_risk_df,
    on="Model"
)

final_results_df = final_results_df.merge(
    high_risk_fnr_df[
        [
            "Model",
            "High-Risk False Negatives",
            "High-Risk False Negative Rate"
        ]
    ],
    on="Model"
)

final_results_df


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,High-Risk Recall,High-Risk False Negatives,High-Risk False Negative Rate
0,XGBoost,0.648936,0.636628,0.644652,0.637035,0.726316,26,0.273684
